# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import pandas as pd

chunks = pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000)
first_chunk = next(chunks)
print(len(first_chunk))

# Just for testing
first_chunk.head()


100000


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A10000012B7CGYKOMPQ4L,000100039X,Adam,"[0, 0]",Spiritually and mentally inspiring! A book tha...,5,Wonderful!,1355616000,"12 16, 2012"
1,A2S166WSCFIFP5,000100039X,"adead_poet@hotmail.com ""adead_poet@hotmail.com""","[0, 2]",This is one my must have books. It is a master...,5,close to god,1071100800,"12 11, 2003"
2,A1BM81XB4QHOA3,000100039X,"Ahoro Blethends ""Seriously""","[0, 0]",This book provides a reflection that you can a...,5,Must Read for Life Afficianados,1390003200,"01 18, 2014"
3,A1MOSTXNIO5MPJ,000100039X,Alan Krug,"[0, 0]",I first read THE PROPHET in college back in th...,5,Timeless for every good and bad time in your l...,1317081600,"09 27, 2011"
4,A2XQ5LZHTD4AFT,000100039X,Alaturka,"[7, 9]",A timeless classic. It is a very demanding an...,5,A Modern Rumi,1033948800,"10 7, 2002"


In [2]:
df = first_chunk
df["user_id"] = pd.factorize(df["reviewerID"])[0]
df["item_id"] = pd.factorize(df["asin"])[0]
df["interaction"] = 1


In [3]:
df = df.sort_values(["user_id", "unixReviewTime"]).copy()

df_test = df.groupby("user_id").tail(1).copy()
df_train = df.drop(df_test.index).copy()

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# MostPopular section

In [4]:
item_popularity = (
    df_train.groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

popular_items = item_popularity.index.tolist()
user_seen_train = df_train.groupby("user_id")["item_id"].apply(set).to_dict()

print(popular_items)

[591, 11, 552, 1552, 244, 601, 42, 1873, 1684, 2040, 172, 321, 367, 1310, 315, 758, 814, 391, 2607, 2447, 817, 2135, 1693, 195, 2146, 280, 13, 2351, 593, 691, 757, 292, 2328, 260, 2005, 2011, 386, 274, 2403, 1779, 190, 373, 320, 2038, 2587, 560, 2421, 2169, 2603, 1871, 2103, 457, 136, 1152, 2151, 2520, 1863, 1459, 2393, 1461, 587, 39, 521, 1472, 0, 2064, 1460, 38, 293, 1643, 2614, 1833, 1838, 437, 441, 2375, 1416, 2525, 2524, 1228, 1981, 477, 1205, 1327, 1368, 2523, 276, 2439, 277, 1158, 2522, 159, 1692, 2506, 509, 1988, 1995, 2440, 180, 142, 819, 249, 1581, 2157, 400, 2254, 165, 1058, 2554, 1178, 2012, 1900, 1332, 2325, 219, 833, 2108, 574, 294, 1359, 1296, 364, 575, 325, 1880, 2615, 878, 2019, 415, 2576, 1153, 2359, 1477, 1093, 2102, 553, 446, 455, 635, 1361, 1443, 1832, 1336, 2058, 1805, 745, 2118, 1525, 1388, 1331, 2162, 2109, 2110, 160, 2153, 2507, 2178, 241, 1257, 569, 576, 1656, 69, 1269, 409, 716, 2635, 2424, 1284, 317, 1735, 1734, 1341, 1865, 2211, 1276, 1732, 436, 2390, 2304,

In [5]:
import math


def recommend_most_popular(user_id, popular_items, user_seen_train, k=10):
    seen = user_seen_train.get(user_id, set())
    recs = []
    for item in popular_items:
        if item not in seen:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0

In [6]:
K = 10
recalls = []
ndcgs = []

for _, row in df_test.iterrows():
    user_id = row["user_id"]
    ground_truth_item = row["item_id"]

    recs = recommend_most_popular(user_id, popular_items, user_seen_train, k=K)

    recalls.append(recall_at_k(recs, ground_truth_item))
    ndcgs.append(ndcg_at_k(recs, ground_truth_item))

print(f"Users evaluated: {len(recalls)}")
# reccomend 10 items and see in what percentage of users that item is in the top 10.
print(f"Recall@{K}: {sum(recalls)/len(recalls):.4f}")
# same but looks at ranking qaulity
print(f"NDCG@{K}: {sum(ndcgs)/len(ndcgs):.4f}")

Users evaluated: 68136
Recall@10: 0.2004
NDCG@10: 0.1069


## BRP

In [16]:
import math
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

# Load Data
chunks = pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000)
df = next(chunks).copy()

df = df[["reviewerID", "asin", "unixReviewTime"]].copy()
df["interaction"] = 1.0

# Proper factorization
df["user_id"] = pd.factorize(df["reviewerID"])[0]
df["item_id"] = pd.factorize(df["asin"])[0]

# Leave-One-Out Split
df = df.sort_values(["user_id", "unixReviewTime"]).copy()

df_test = df.groupby("user_id").tail(1).copy()
df_train = df.drop(df_test.index).copy()

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Remove users that lost all train interactions
train_user_counts = df_train["user_id"].value_counts()
valid_users = set(train_user_counts.index)

df_train = df_train[df_train["user_id"].isin(valid_users)].copy()
df_test = df_test[df_test["user_id"].isin(valid_users)].copy()

# Remove test items unseen in train
valid_items = set(df_train["item_id"].unique())
df_test = df_test[df_test["item_id"].isin(valid_items)].copy()

# Remap IDs
user_ids = sorted(df_train["user_id"].unique())
item_ids = sorted(df_train["item_id"].unique())

user_map = {old: new for new, old in enumerate(user_ids)}
item_map = {old: new for new, old in enumerate(item_ids)}

df_train["user_id"] = df_train["user_id"].map(user_map)
df_train["item_id"] = df_train["item_id"].map(item_map)

df_test["user_id"] = df_test["user_id"].map(user_map)
df_test["item_id"] = df_test["item_id"].map(item_map)

df_train = df_train.dropna().copy()
df_test = df_test.dropna().copy()

df_train["user_id"] = df_train["user_id"].astype(int)
df_train["item_id"] = df_train["item_id"].astype(int)
df_test["user_id"] = df_test["user_id"].astype(int)
df_test["item_id"] = df_test["item_id"].astype(int)

num_users = int(df_train["user_id"].max()) + 1
num_items = int(df_train["item_id"].max()) + 1

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print("Users:", num_users, "Items:", num_items)

# Build Sparse Matrix
user_items_csr = csr_matrix(
    (
        df_train["interaction"].astype(np.float32),
        (df_train["user_id"], df_train["item_id"])
    ),
    shape=(num_users, num_items)
)

item_users_csr = user_items_csr.T.tocsr()

model = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.05,
    regularization=0.01,
    iterations=100,
    random_state=42
)

model.fit(item_users_csr, show_progress=True)

print("Model user factors:", model.user_factors.shape)
print("Model item factors:", model.item_factors.shape)

# Evaluation
def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0

K = 10
recalls = []
ndcgs = []

max_user_index = model.user_factors.shape[0] - 1
df_test = df_test[df_test["user_id"] <= max_user_index].copy()

for _, row in df_test.iterrows():
    user_id = int(row["user_id"])
    ground_truth_item = int(row["item_id"])

    recs, scores = model.recommend(
        userid=user_id,
        user_items=user_items_csr[user_id],
        N=K,
        filter_already_liked_items=True
    )

    recs = recs.tolist()
    recalls.append(recall_at_k(recs, ground_truth_item))
    ndcgs.append(ndcg_at_k(recs, ground_truth_item))

print("\n=== BPR-MF Results ===")
print(f"Users evaluated: {len(recalls)}")
print(f"Recall@{K}: {np.mean(recalls):.4f}")
print(f"NDCG@{K}: {np.mean(ndcgs):.4f}")

Train shape: (31864, 6)
Test shape: (14781, 6)
Users: 14975 Items: 2510


100%|██████████| 100/100 [00:01<00:00, 64.96it/s, train_auc=99.93%, skipped=1.88%]


Model user factors: (2510, 65)
Model item factors: (14975, 65)

=== BPR-MF Results ===
Users evaluated: 2487
Recall@10: 0.0012
NDCG@10: 0.0007


# Lights GCN